# Lingo-QA Fine-Tuning Notebook

This notebook demonstrates fine-tuning Qwen2-VL-2B-Instruct on the LingoQA autonomous driving dataset using LoRA adapters.

## Workflow
1. Load base model and processor
2. Load LingoQA dataset with multi-frame images
3. Configure LoRA for efficient fine-tuning
4. Set up training with SFTTrainer
5. Train and save checkpoints

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
from peft import get_peft_model
from trl import SFTConfig, SFTTrainer
from dataset import LingoQADataset, QwenVLBatchCollator
from utils import clear_memory, get_lora_config, generate_text, Config

# Load config.yaml
config = Config()

/root/lingo-qa-fine-tune/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Setup and Load Model

In [ ]:
model_id = config.get("model.id")
device = torch.device(config.get("model.device", "cuda"))
dtype = torch.bfloat16 if config.get("model.dtype") == 'bfloat16' else torch.float16

In [ ]:
# Load model and tokenizer
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", torch_dtype=dtype
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.10it/s]


In [ ]:
# Use max_pixels from config if present
max_pixels = config.get("model.max_pixels", 322*511)
processor = Qwen2VLProcessor.from_pretrained(model_id, max_pixels=max_pixels)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 11184.81it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 13797.05it/s]


## 2. Inference Example (Optional)

In [5]:
import pandas as pd
df = pd.read_parquet("./data/val/val.parquet")

In [6]:
df

,question_id,segment_id,images,question,answer
0,1a938d25604410ccd63b60285919eaec,f2e4286e94457b8605069190e29f955a,[images/val/f2e4286e94457b8605069190e29f955a/0...,"Is there a traffic light? If yes, what color i...","Yes, green."
1,1a938d25604410ccd63b60285919eaec,f2e4286e94457b8605069190e29f955a,[images/val/f2e4286e94457b8605069190e29f955a/0...,"Is there a traffic light? If yes, what color i...","Yes, a temporary traffic light. It is showing ..."
2,49cd6149f02ec01ee03a6eb8287ab2df,f2e4286e94457b8605069190e29f955a,[images/val/f2e4286e94457b8605069190e29f955a/0...,How many pedestrians can be seen?,None.
3,49cd6149f02ec01ee03a6eb8287ab2df,f2e4286e94457b8605069190e29f955a,[images/val/f2e4286e94457b8605069190e29f955a/0...,How many pedestrians can be seen?,No pedestrian can be seen.
4,c6050fe51d3408b1eaef27e08e7f1956,f2e4286e94457b8605069190e29f955a,[images/val/f2e4286e94457b8605069190e29f955a/0...,What is the current action and its justificati...,"The car starts and moves right, because the ve..."
...,...,...,...,...,...
135,61804c43c7287cd9045a67b7b7501af5,fbdf56c8cf8500903a2b6b88f84522f0,[images/val/fbdf56c8cf8500903a2b6b88f84522f0/0...,Is there any vehicle in the lane to your left?...,No.
136,a6df923bb4967b7d689f7cd253a1dd03,fbdf56c8cf8500903a2b6b88f84522f0,[images/val/fbdf56c8cf8500903a2b6b88f84522f0/0...,Is it safe to accelerate considering the curre...,"Yes, there is an increasing amount of space be..."
137,a6df923bb4967b7d689f7cd253a1dd03,fbdf56c8cf8500903a2b6b88f84522f0,[images/val/fbdf56c8cf8500903a2b6b88f84522f0/0...,Is it safe to accelerate considering the curre...,"Yes, the vehicles ahead are accelerating and t..."
138,ee746fd004096aeccdef57c5ed6ca3be,fbdf56c8cf8500903a2b6b88f84522f0,[images/val/fbdf56c8cf8500903a2b6b88f84522f0/0...,What is the current action and its justificati...,"The car accelerates, in order to match the spe..."


In [13]:
images

['data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/0.jpg',
 'data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/1.jpg',
 'data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/2.jpg',
 'data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/3.jpg',
 'data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/4.jpg']

In [8]:
system_message = ""

images = [
    "data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/" + f"{i}.jpg"
    for i in range(5)
]


sample = {
        "images": images,
        "messages": [
            {
                "role": "system",
                "content": [
                    {"type": "text", "text": system_message + ' Question: Are there any pedestrians crossing the road? If yes, how many?'}
                ] + [
                     {"type": "image", "image": img}
                    for img in images
                ],
            },            
        ]
}

In [10]:
df.iloc[10][["images", "question", "answer"]].tolist()

[array(['images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/0.jpg',
        'images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/1.jpg',
        'images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/2.jpg',
        'images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/3.jpg',
        'images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/4.jpg'], dtype=object),
 'Are there any pedestrians crossing the road? If yes, how many?',
 'Yes, 2: one close to the car, one further away.']

In [28]:
images[0]

'data/val/images/val/0cb55ed0d4ce02a27a2647c97c2ab6e8/0.jpg'

In [ ]:
# Generate text with the model
output_text = generate_text(model, processor, sample["messages"], device, max_new_tokens=100)

In [45]:
output_text

['The image shows a street scene with several pedestrians crossing the road. There are at least four pedestrians visible in the image.']

## 3. Prepare Training Data

In [ ]:
# Load datasets
system_message = "You are an autonomous driving agent. Answer the question concisely without uncesessary details. Question: "
train_dataset = LingoQADataset("./data/val/val.parquet", image_dir="./data/val/")
val_dataset = LingoQADataset("./data/val/val.parquet", image_dir="./data/val/")

# Create batch collator
collator = QwenVLBatchCollator(processor, system_message)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 13842.59it/s]


{'input_ids': tensor([151643, 151643, 151643,  ...,     13, 151645,    198]), 'attention_mask': tensor([0, 0, 0,  ..., 1, 1, 1]), 'pixel_values': tensor([0.0471, 0.1639, 0.0763,  ..., 0.4679, 0.4253, 0.4395]), 'image_grid_thw': tensor([ 1, 22, 36]), 'labels': tensor([  -100,   -100,   -100,  ...,     13, 151645,    198])}


In [ ]:
# Configure LoRA and apply PEFT using config
peft_config = get_lora_config(config=config)
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 18,735,104 || all params: 2,227,720,704 || trainable%: 0.8410


## 4. Configure LoRA

## 5. Train Model

In [ ]:
# Configure training arguments from config
training_args = SFTConfig(
    output_dir=config.get('training.output_dir'),
    num_train_epochs=config.get('training.num_train_epochs'),
    per_device_train_batch_size=config.get('training.per_device_train_batch_size'),
    per_device_eval_batch_size=config.get('training.per_device_eval_batch_size'),
    gradient_accumulation_steps=config.get('training.gradient_accumulation_steps'),
    gradient_checkpointing_kwargs=config.get('training.gradient_checkpointing_kwargs'),
    max_length=config.get('training.max_length'),
    optim=config.get('training.optim'),
    learning_rate=config.get('training.learning_rate'),
    logging_steps=config.get('training.logging_steps'),
    save_strategy=config.get('training.save_strategy'),
    save_steps=config.get('training.save_steps'),
    bf16=config.get('training.bf16'),
    max_grad_norm=config.get('training.max_grad_norm'),
    warmup_ratio=config.get('training.warmup_ratio'),
    report_to=config.get('training.report_to'),
)

# Create trainer
trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

In [ ]:
# Start training
trainer.train()

## 6. Cleanup (Optional)

In [ ]:
# Clean up GPU memory
clear_memory()

GPU allocated memory: 6.19 GB
GPU reserved memory: 6.25 GB
